# CSC3310 Final Project - A* Search Algorithm

In [1]:
import math

In [2]:
class Graph:
    def __init__(self):
        self.nodes = {}
        self.adjacency = {}

    def add_node(self, node_id, coordinates):
        self.nodes[node_id] = coordinates
        self.adjacency[node_id] = []

    def add_edge(self, from_id, to_id, distance_meters, transport_modes, has_stairs=False, bidirectional=True):
        edge = {
            "from": from_id,
            "to": to_id,
            "distance_meters": distance_meters,
            "transport_modes": transport_modes,
            "has_stairs": has_stairs,
            "bidirectional": bidirectional,
        }
        self.adjacency[from_id].append(edge)
        self.adjacency[to_id].append(edge)

In [3]:
def haversine_distance(a, b):
    lon1, lat1, lon2, lat2 = map(math.radians, (*a, *b))
    h = math.sin((lat2 - lat1) / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371000 * math.asin(math.sqrt(h))

In [4]:
def find_route(graph, start_node_id, end_node_id, options):
    start_node = graph.nodes.get(start_node_id)
    end_node = graph.nodes.get(end_node_id)

    if not start_node or not end_node:
        return None
    if start_node_id == end_node_id:
        return [start_node]

    open_set = {}
    closed_set = set()
    came_from = {}
    g_scores = {}

    g_scores[start_node_id] = 0
    open_set[start_node_id] = {
        "g_score": 0,
        "f_score": haversine_distance(start_node, end_node),
    }

    while open_set:
        current_id = get_lowest_f_score(open_set)

        if current_id == end_node_id:
            return reconstruct_path(graph, came_from, current_id)

        del open_set[current_id]
        closed_set.add(current_id)

        current_g = g_scores.get(current_id, math.inf)

        for edge in graph.adjacency[current_id]:
            if options["transport_mode"] not in edge["transport_modes"]:
                continue

            if options.get("no_stairs") and edge["has_stairs"]:
                continue

            if not edge["bidirectional"] and edge["to"] == current_id:
                continue

            neighbor_id = edge["to"] if edge["from"] == current_id else edge["from"]

            if neighbor_id in closed_set:
                continue

            neighbor_node = graph.nodes.get(neighbor_id)
            if not neighbor_node:
                continue

            tentative_g = current_g + edge["distance_meters"]
            known_g = g_scores.get(neighbor_id, math.inf)

            if tentative_g < known_g:
                came_from[neighbor_id] = current_id
                g_scores[neighbor_id] = tentative_g

                h = haversine_distance(neighbor_node, end_node)

                open_set[neighbor_id] = {
                    "g_score": tentative_g,
                    "f_score": tentative_g + h,
                }

    return None


def get_lowest_f_score(open_set):
    lowest_id = ""
    lowest_score = math.inf

    for id, node in open_set.items():
        if node["f_score"] < lowest_score:
            lowest_score = node["f_score"]
            lowest_id = id

    return lowest_id


def reconstruct_path(graph, came_from, end_node_id):
    path = []
    current = end_node_id

    while current in came_from:
        path.insert(0, graph.nodes[current])
        current = came_from[current]

    path.insert(0, graph.nodes[current])

    return path